# 03 — From GRIB to Tensogram

<sup>(C) Copyright 2026- ECMWF and individual contributors. Licensed under the Apache Licence, Version 2.0.</sup>

**What you will learn**

- Convert a real ECMWF opendata GRIB2 message into Tensogram wire format with one Python call.
- Two entry points: `tensogram.convert_grib(path)` (filesystem) and `tensogram.convert_grib_buffer(bytes)` (in-memory).
- Cross-verify the round-trip against what ecCodes reads from the source.
- Choose a compression pipeline when converting: exact `none`, lossless `simple_packing + szip` (GRIB-equivalent), or compact `simple_packing + zstd`.
- Preserve all ecCodes namespace keys with `preserve_all_keys=True`.

**Prerequisites**

- tensogram Python bindings built with `--features grib` — see [`README.md`](README.md).
- `libeccodes` installed at the OS level.
- Real GRIB fixture already committed in the repo: `rust/tensogram-grib/testdata/2t.grib2` (IFS 0.25° operational, 2026-04-04 00z, step 0, 2m temperature).

> **No network required.** The GRIB fixture is committed in-tree — see `rust/tensogram-grib/testdata/download.sh` for how it was originally fetched from `data.ecmwf.int`.

In [ ]:
import matplotlib

matplotlib.use("Agg")

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import tensogram

if not getattr(tensogram, "__has_grib__", False):
    print(
        "ERROR: tensogram was built without the 'grib' feature.\n"
        "Rebuild with: cd python/bindings && maturin develop --features grib\n"
        "(Requires libeccodes installed at the OS level.)",
        file=sys.stderr,
    )
    sys.exit(0)  # clean exit so the notebook still runs to completion in CI

# Walk up from the notebook to find the repo root, then the GRIB fixture.
HERE = Path.cwd()
for p in [HERE, *HERE.parents]:
    candidate = p / "rust" / "tensogram-grib" / "testdata" / "2t.grib2"
    if candidate.exists():
        GRIB_PATH = candidate
        break
else:
    raise FileNotFoundError("could not locate 2t.grib2 — run from the tensogram repo")

print(f"GRIB fixture: {GRIB_PATH}")
print(f"size:         {GRIB_PATH.stat().st_size:,} bytes")

## 1. Simplest conversion: file path → Tensogram bytes

`tensogram.convert_grib(path)` is the most convenient entry point. It returns a list of `bytes`, one per output Tensogram message. The default `grouping="merge_all"` packs all GRIB messages from the input file into a single Tensogram message with N data objects.

In [ ]:
messages = tensogram.convert_grib(str(GRIB_PATH))
print(f"produced {len(messages)} Tensogram message(s)")
print(f"first message is {len(messages[0]):,} bytes "
      f"({len(messages[0]) / GRIB_PATH.stat().st_size:.1f}× the raw GRIB size)")

# The default pipeline is "no encoding, no compression" — the payload
# is float64. GRIB's own packing is unpacked by ecCodes during the read,
# so this is *bigger* than the GRIB. We'll fix that in a moment.
print("(default pipeline is uncompressed float64 — see section 4 for size vs fidelity tuning)")

## 2. Inspect the output

The Tensogram message carries:
- The decoded **values** as a 2D latitude × longitude `float64` array.
- A `base[0]["mars"]` sub-map with the MARS keys ecCodes surfaces for this message.
- An xxh3-64 integrity hash computed during encoding.

In [ ]:
meta, objects = tensogram.decode(messages[0])
desc, temperature = objects[0]

print(f"shape      = {temperature.shape}        (721 x 1440 on the 0.25° global grid)")
print(f"dtype      = {temperature.dtype}")
print(f"value range = [{temperature.min():.2f}, {temperature.max():.2f}] K")
# v3: the per-object hash moved from the CBOR descriptor to
# the frame footer's inline slot (plans/WIRE_FORMAT.md §2.4).
# Use `tensogram.validate(msg, level="checksum")` for integrity.
print()
print("mars keys:")
for key, value in sorted(meta.base[0]["mars"].items()):
    print(f"  {key:<10} = {value!r}")

## 3. Visualise

A quick world map of the decoded 2m temperature field.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
# IFS 0.25° global: 721 latitudes (90°N → 90°S), 1440 longitudes (0° → 360°).
im = ax.imshow(
    temperature - 273.15,  # show in Celsius
    extent=[0, 360, -90, 90],
    origin="upper",
    cmap="RdYlBu_r",
    aspect="auto",
    vmin=-60, vmax=40,
)
ax.set_xlabel("longitude (deg)")
ax.set_ylabel("latitude (deg)")
ax.set_title(
    f"2m temperature — ECMWF IFS 0.25° operational, "
    f"{meta.base[0]['mars']['date']} {meta.base[0]['mars']['time']}z step={meta.base[0]['mars']['step']}"
)
plt.colorbar(im, ax=ax, label="°C")
plt.tight_layout()
plt.show()

## 4. Tune the pipeline for size vs fidelity

The raw `convert_grib(path)` call kept the float64 values uncompressed. For real storage you'll want a compression pipeline. Three useful presets:

| Pipeline | Loss | Typical size | Use case |
|---|---|---|---|
| `simple_packing` 16-bit + `szip` | quantization | ~3% of raw f64 | GRIB-interoperable (byte-for-byte equivalent to `grid_ccsds`) |
| `simple_packing` 24-bit + `zstd` | tiny | ~5–8% of raw f64 | sub-mK fidelity, better ratio than GRIB |
| `shuffle` + `zstd` | none (lossless) | ~15–30% | when you need exact values |


In [ ]:
pipelines = [
    ("baseline (uncompressed f64)", {}),
    ("simple_packing 16 + szip",
     dict(encoding="simple_packing", bits=16, compression="szip")),
    ("simple_packing 24 + zstd-3",
     dict(encoding="simple_packing", bits=24, compression="zstd", compression_level=3)),
    ("shuffle + zstd-3 (lossless)",
     dict(filter="shuffle", compression="zstd", compression_level=3)),
]

baseline_size = None
for name, kwargs in pipelines:
    msgs = tensogram.convert_grib(str(GRIB_PATH), **kwargs)
    size = len(msgs[0])
    if baseline_size is None:
        baseline_size = size
    _meta, objs = tensogram.decode(msgs[0])
    decoded = objs[0][1]
    linf = float(np.abs(decoded - temperature).max())
    print(f"{name:<40} {size:>8,} bytes  ({size/baseline_size*100:5.1f}% of raw)  "
          f"Linf={linf:.4f} K")

## 5. In-memory conversion: `convert_grib_buffer`

Sometimes you do not have (or do not want to create) a file on disk — you've just fetched a GRIB message via an HTTP byte-range request, or pulled it from a cache. `convert_grib_buffer` accepts any Python bytes-like object.

In [ ]:
# Load the GRIB bytes into memory — imagine these came from
# `requests.get(url, headers={"Range": "bytes=74573515-75234113"}).content`.
grib_bytes = GRIB_PATH.read_bytes()
print(f"grib_bytes: {type(grib_bytes).__name__}  ({len(grib_bytes):,} bytes)")

from_buffer = tensogram.convert_grib_buffer(grib_bytes,
                                             encoding="simple_packing",
                                             bits=16,
                                             compression="szip")
from_file = tensogram.convert_grib(str(GRIB_PATH),
                                    encoding="simple_packing",
                                    bits=16,
                                    compression="szip")
print(f"buffer path produced: {len(from_buffer[0]):,} bytes")
print(f"file path produced:   {len(from_file[0]):,} bytes")

# The encoded bytes can differ (each call stamps a fresh uuid and timestamp
# in _reserved_), but the *decoded payloads* must be bit-identical.
_, objs_file = tensogram.decode(from_file[0])
_, objs_buffer = tensogram.decode(from_buffer[0])
np.testing.assert_array_equal(objs_file[0][1], objs_buffer[0][1])
print("\ndecoded payloads match bit-for-bit")

`convert_grib_buffer` accepts any Python bytes-like object — `bytes`, `bytearray`, `memoryview`, `numpy.uint8[:]`. This works because the Python extraction goes through the buffer protocol.

In [ ]:
for label, obj in [
    ("bytes",       grib_bytes),
    ("bytearray",   bytearray(grib_bytes)),
    ("memoryview",  memoryview(grib_bytes)),
    ("numpy.uint8", np.frombuffer(grib_bytes, dtype=np.uint8)),
]:
    msgs = tensogram.convert_grib_buffer(obj)
    print(f"{label:<14} -> {len(msgs[0]):,} bytes")

## 6. `preserve_all_keys=True` — lift every ecCodes namespace

By default only MARS keys are extracted into `base[i]["mars"]`. Pass `preserve_all_keys=True` to also lift the full ecCodes namespace (`ls`, `geography`, `time`, `vertical`, `parameter`, `statistics`) into `base[i]["grib"]`. Useful for round-tripping back to a GRIB writer or for detailed metadata introspection.

In [ ]:
msgs = tensogram.convert_grib(str(GRIB_PATH), preserve_all_keys=True)
meta = tensogram.decode_metadata(msgs[0])

print("namespaces populated under base[0]['grib']:")
for ns, keys in meta.base[0]["grib"].items():
    print(f"  {ns:<12} ({len(keys)} keys): {sorted(keys.keys())[:5]}{'...' if len(keys) > 5 else ''}")

## 7. File API sub-section — putting it on disk

Once you have a `.tgm` file, the `TensogramFile` class gives you a lazy, indexable view over all messages inside — no need to eagerly decode everything.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    out = Path(tmpdir) / "2t.tgm"
    # Build a compact .tgm from the GRIB.
    with out.open("wb") as fh:
        for msg in tensogram.convert_grib(str(GRIB_PATH),
                                          encoding="simple_packing", bits=16,
                                          compression="szip"):
            fh.write(msg)
    print(f"wrote {out}  ({out.stat().st_size:,} bytes)")

    # Read it back lazily.
    with tensogram.TensogramFile.open(str(out)) as f:
        print(f"message count:   {len(f)}")
        # f[i] returns the full decoded Message for message i.
        msg0 = f[0]
        print(f"message[0] has {len(msg0.objects)} object(s)")
        _desc, data = msg0.objects[0]
        print(f"object[0] shape = {data.shape}  dtype = {data.dtype}")

## Where to go next

- [`04_from_netcdf_to_tensogram.ipynb`](04_from_netcdf_to_tensogram.ipynb) — same end-to-end story for NetCDF inputs, with xarray integration.
- [`05_validation_and_parallelism.ipynb`](05_validation_and_parallelism.ipynb) — validate the output, scale encoding across threads.
- [`../../docs/src/grib/overview.md`](../../docs/src/grib/overview.md) — the conversion reference.